In [ ]:
tableName = "table1"

In [4]:
from pyspark.sql import functions as F 
from datetime import datetime 

StatementMeta(, 9e7f9f57-f3b9-4c3b-a41f-6e7d5188e44b, 6, Finished, Available, Finished, False)

In [5]:

tables = { 
    'bronze_transactions': {'min_rows': 1000000, 'key_col': 'nameOrig'}, 
    'bronze_customers':    {'min_rows': 10000,   'key_col': 'customer_id'}, 
    'bronze_products':     {'min_rows': 5,       'key_col': 'product_id'}, 
    'bronze_branches':     {'min_rows': 10,      'key_col': 'branch_id'}, 
}

StatementMeta(, 9e7f9f57-f3b9-4c3b-a41f-6e7d5188e44b, 7, Finished, Available, Finished, False)

In [6]:
validation_results = []

for table_name, config in tables.items():

    try:

        df = spark.read.format("delta").load(f"Tables/dbo/{table_name}")

        row_count = df.count()

        null_count = (
            df.filter(F.col(config["key_col"]).isNull())
            .count()
        )

        dupe_count = (
            df.groupBy(config["key_col"])
            .count()
            .filter("count > 1")
            .count()
        )

        status = (
            "PASS"
            if row_count >= config["min_rows"] and null_count == 0
            else "FAIL"
        )

        validation_results.append({
            "table": table_name,
            "row_count": row_count,
            "null_keys": null_count,
            "duplicates": dupe_count,
            "status": status
        })

        print(
            f"{status}: {table_name} | "
            f"Rows={row_count:,} | "
            f"Nulls={null_count} | "
            f"Dupes={dupe_count}"
        )

    except Exception as e:

        print(f"FAIL: {table_name} - Error: {e}")

        validation_results.append({
            "table": table_name,
            "status": "ERROR",
            "error": str(e)
        })
# Save validation log as a Delta table
val_df = spark.createDataFrame(validation_results)

val_df = val_df.withColumn(
    "validated_at",
    F.current_timestamp()
)

val_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze_validation_log")


StatementMeta(, 9e7f9f57-f3b9-4c3b-a41f-6e7d5188e44b, 8, Finished, Available, Finished, False)

PASS: bronze_transactions | Rows=6,362,620 | Nulls=0 | Dupes=9298
PASS: bronze_customers | Rows=50,000 | Nulls=0 | Dupes=0
PASS: bronze_products | Rows=10 | Nulls=0 | Dupes=0
PASS: bronze_branches | Rows=200 | Nulls=0 | Dupes=0
